In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_RAW = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')
RAINFALL_DIR = DATA_RAW / 'RainfallData'

In [3]:
rainfall_df = pd.read_csv(DATA_PROCESSED / 'combined_rainfall_data.csv')
rainfall_df['Date'] = pd.to_datetime(rainfall_df['Date'])
rainfall_df = rainfall_df.sort_values(['Region', 'Date']).reset_index(drop=True)

print(f"Loaded {len(rainfall_df):,} records")
print(f"Date range: {rainfall_df['Date'].min().date()} to {rainfall_df['Date'].max().date()}")
print(f"Regions: {sorted(rainfall_df['Region'].unique())}")
rainfall_df.head()

Loaded 67,570 records
Date range: 1983-01-01 to 2019-12-31
Regions: ['Region1', 'Region2', 'Region3', 'Region4', 'Region5']


,Year,Month,Day,CHIRPS,TAMSAT,Mean,Region,Date,Rainy_Day
0,1983,1,1,0.000000,0.368836,0.184418,Region1,1983-01-01,False
1,1983,1,2,0.000000,0.088224,0.044112,Region1,1983-01-02,False
2,1983,1,3,0.000000,0.108656,0.054328,Region1,1983-01-03,False
3,1983,1,4,0.811411,1.085505,0.948458,Region1,1983-01-04,False
4,1983,1,5,1.025914,0.658789,0.842351,Region1,1983-01-05,False


In [4]:
def add_rolling_features(df, rainfall_col='Mean', windows=[7, 14, 30]):
    """Add rolling rainfall statistics for each region."""
    result = df.copy()
    
    for window in windows:
        # Rolling sum
        result[f'rainfall_sum_{window}d'] = (
            result.groupby('Region')[rainfall_col]
            .transform(lambda x: x.rolling(window, min_periods=1).sum())
        )
        
        # Rolling mean
        result[f'rainfall_avg_{window}d'] = (
            result.groupby('Region')[rainfall_col]
            .transform(lambda x: x.rolling(window, min_periods=1).mean())
        )
        
        # Rolling max (intensity)
        result[f'rainfall_max_{window}d'] = (
            result.groupby('Region')[rainfall_col]
            .transform(lambda x: x.rolling(window, min_periods=1).max())
        )
        
        # Rolling std (variability)
        result[f'rainfall_std_{window}d'] = (
            result.groupby('Region')[rainfall_col]
            .transform(lambda x: x.rolling(window, min_periods=1).std())
        ).fillna(0)
        
        # Rainy days count in window
        result[f'rainy_days_{window}d'] = (
            result.groupby('Region')[rainfall_col]
            .transform(lambda x: (x > 1.0).rolling(window, min_periods=1).sum())
        )
    
    return result

rainfall_df = add_rolling_features(rainfall_df)
print("Rolling features added:")
print([c for c in rainfall_df.columns if 'rainfall_' in c or 'rainy_days' in c])

Rolling features added:
['rainfall_sum_7d', 'rainfall_avg_7d', 'rainfall_max_7d', 'rainfall_std_7d', 'rainy_days_7d', 'rainfall_sum_14d', 'rainfall_avg_14d', 'rainfall_max_14d', 'rainfall_std_14d', 'rainy_days_14d', 'rainfall_sum_30d', 'rainfall_avg_30d', 'rainfall_max_30d', 'rainfall_std_30d', 'rainy_days_30d']


In [5]:
def add_dry_spell_features(df, rainfall_col='Mean', threshold=1.0):
    """Calculate dry spell duration and related features."""
    result = df.copy()
    
    # Is dry day (rainfall below threshold)
    result['is_dry_day'] = (result[rainfall_col] < threshold).astype(int)
    
    # Current dry spell length (consecutive dry days)
    def calc_dry_spell_length(group):
        is_dry = group['is_dry_day'].values
        dry_spell = np.zeros(len(is_dry), dtype=int)
        
        for i in range(len(is_dry)):
            if is_dry[i]:
                dry_spell[i] = dry_spell[i-1] + 1 if i > 0 else 1
            else:
                dry_spell[i] = 0
        
        return pd.Series(dry_spell, index=group.index)
    
    result['dry_spell_length'] = result.groupby('Region', group_keys=False).apply(calc_dry_spell_length)
    
    # Max dry spell in last 30 days
    result['max_dry_spell_30d'] = (
        result.groupby('Region')['dry_spell_length']
        .transform(lambda x: x.rolling(30, min_periods=1).max())
    )
    
    # Days since last significant rain (>5mm)
    def days_since_rain(group, threshold=5.0):
        rain = group[rainfall_col].values
        days_since = np.zeros(len(rain), dtype=int)
        
        last_rain = -1
        for i in range(len(rain)):
            if rain[i] >= threshold:
                last_rain = i
            days_since[i] = i - last_rain if last_rain >= 0 else i + 1
        
        return pd.Series(days_since, index=group.index)
    
    result['days_since_rain_5mm'] = result.groupby('Region', group_keys=False).apply(
        lambda g: days_since_rain(g, threshold=5.0)
    )
    
    return result

rainfall_df = add_dry_spell_features(rainfall_df)
print("Dry spell features added:")
print([c for c in rainfall_df.columns if 'dry' in c.lower() or 'since' in c.lower()])

Dry spell features added:
['is_dry_day', 'dry_spell_length', 'max_dry_spell_30d', 'days_since_rain_5mm']


## Rainy Season Onset/Cessation Detection

Using standard agronomic definitions:

- **Onset**: First occurrence of at least 20mm in 3 days, followed by no dry spell >7 days in next 30 days
- **Cessation**: Last significant rainfall before prolonged dry period


In [6]:
def detect_season_onset(df, rainfall_col='Mean'):
    """
    Detect rainy season onset for each year and region.
    Simplified criteria: 20mm cumulative in 3 days after a dry spell.
    """
    results = []
    
    for (region, year), group in df.groupby(['Region', 'Year']):
        group = group.sort_values('Date')
        
        rainfall = group[rainfall_col].values
        doy = group['Date'].dt.dayofyear.values
        dates = group['Date'].values
        
        onset_doy = None
        cessation_doy = None
        
        # Find onset (first 3-day period with >20mm after day 60)
        for i in range(60, min(200, len(rainfall) - 3)):  # Search Mar-Jul
            three_day_sum = sum(rainfall[i:i+3])
            if three_day_sum >= 20:
                # Check no 7-day dry spell in next 20 days
                if i + 23 < len(rainfall):
                    next_20_days = rainfall[i+3:i+23]
                    max_consecutive_dry = 0
                    current_dry = 0
                    for r in next_20_days:
                        if r < 1:
                            current_dry += 1
                            max_consecutive_dry = max(max_consecutive_dry, current_dry)
                        else:
                            current_dry = 0
                    
                    if max_consecutive_dry < 7:
                        onset_doy = doy[i]
                        break
        
        # Find cessation (last 3-day period with >10mm before day 320)
        for i in range(min(320, len(rainfall) - 3), 200, -1):  # Search backward from Nov
            three_day_sum = sum(rainfall[i:i+3])
            if three_day_sum >= 10:
                cessation_doy = doy[i+2] if i+2 < len(doy) else doy[i]
                break
        
        results.append({
            'Region': region,
            'Year': year,
            'onset_doy': onset_doy,
            'cessation_doy': cessation_doy,
            'season_length': (cessation_doy - onset_doy) if (onset_doy and cessation_doy) else None
        })
    
    return pd.DataFrame(results)

season_dates = detect_season_onset(rainfall_df)
print("Season onset/cessation detected:")
print(season_dates.groupby('Region')[['onset_doy', 'cessation_doy', 'season_length']].mean().round(1))

Season onset/cessation detected:
         onset_doy  cessation_doy  season_length
Region                                          
Region1       88.2          320.3          232.1
Region2       77.1          322.2          245.0
Region3       69.9          322.5          252.6
Region4       79.6          322.4          242.8
Region5       75.7          321.7          246.0


In [ ]:
rainfall_df = rainfall_df.merge(season_dates, on=['Region', 'Year'], how='left')

# Add relative position in season
rainfall_df['doy'] = rainfall_df['Date'].dt.dayofyear
rainfall_df['days_from_onset'] = rainfall_df['doy'] - rainfall_df['onset_doy']
rainfall_df['days_to_cessation'] = rainfall_df['cessation_doy'] - rainfall_df['doy']

# In-season flag
rainfall_df['in_season'] = (
    (rainfall_df['doy'] >= rainfall_df['onset_doy']) & 
    (rainfall_df['doy'] <= rainfall_df['cessation_doy'])
).astype(int)

print("Season position features added")

Season position features added


## Temporal Encodings

Cyclical encoding preserves the circular nature of time.


In [8]:
def add_temporal_features(df):
    """Add cyclical temporal encodings."""
    result = df.copy()
    
    # Day of year (1-365/366)
    result['doy'] = result['Date'].dt.dayofyear
    
    # Cyclical encoding for day of year
    result['doy_sin'] = np.sin(2 * np.pi * result['doy'] / 365)
    result['doy_cos'] = np.cos(2 * np.pi * result['doy'] / 365)
    
    # Month cyclical
    result['month_sin'] = np.sin(2 * np.pi * result['Month'] / 12)
    result['month_cos'] = np.cos(2 * np.pi * result['Month'] / 12)
    
    # Dekad (10-day period) - common in agricultural applications
    result['dekad'] = ((result['Day'] - 1) // 10 + 1).clip(upper=3)  # 1, 2, or 3
    result['dekad_of_year'] = (result['Month'] - 1) * 3 + result['dekad']  # 1-36
    
    # Week of year
    result['week_of_year'] = result['Date'].dt.isocalendar().week.astype(int)
    
    return result

rainfall_df = add_temporal_features(rainfall_df)
print("Temporal features added:")
print([c for c in rainfall_df.columns if 'doy' in c or 'month_' in c or 'dekad' in c or 'week' in c])

Temporal features added:
['onset_doy', 'cessation_doy', 'doy', 'doy_sin', 'doy_cos', 'month_sin', 'month_cos', 'dekad', 'dekad_of_year', 'week_of_year']


In [9]:
def add_lag_features(df, rainfall_col='Mean', lags=[1, 7, 14, 30]):
    """Add lagged rainfall values."""
    result = df.copy()
    
    for lag in lags:
        result[f'rainfall_lag_{lag}d'] = (
            result.groupby('Region')[rainfall_col]
            .shift(lag)
        )
    
    # Same day last year (approximate)
    result['rainfall_lag_1y'] = (
        result.groupby('Region')[rainfall_col]
        .shift(365)
    )
    
    return result

rainfall_df = add_lag_features(rainfall_df)
print("Lag features added:")
print([c for c in rainfall_df.columns if 'lag' in c])

Lag features added:
['rainfall_lag_1d', 'rainfall_lag_7d', 'rainfall_lag_14d', 'rainfall_lag_30d', 'rainfall_lag_1y']


In [10]:
def add_cumulative_features(df, rainfall_col='Mean'):
    """Add cumulative rainfall within the growing season."""
    result = df.copy()
    
    # Cumulative rainfall since January 1
    result['cumulative_annual'] = (
        result.groupby(['Region', 'Year'])[rainfall_col]
        .cumsum()
    )
    
    # Cumulative since season onset
    def cumsum_since_onset(group):
        onset = group['onset_doy'].iloc[0]
        if pd.isna(onset):
            return pd.Series(np.nan, index=group.index)
        
        in_season = group['doy'] >= onset
        cumsum = group[rainfall_col].where(in_season, 0).cumsum()
        cumsum = cumsum.where(in_season, 0)
        return cumsum
    
    result['cumulative_season'] = (
        result.groupby(['Region', 'Year'], group_keys=False)
        .apply(cumsum_since_onset)
    )
    
    return result

rainfall_df = add_cumulative_features(rainfall_df)
print("Cumulative features added")

Cumulative features added


## Anomaly Features (Deviation from Normal)


In [11]:
def add_anomaly_features(df, rainfall_col='Mean'):
    """Calculate deviation from long-term climatology."""
    result = df.copy()
    
    # Calculate long-term daily mean by region and day-of-year
    climatology = (
        result.groupby(['Region', 'doy'])[rainfall_col]
        .agg(['mean', 'std'])
        .reset_index()
    )
    climatology.columns = ['Region', 'doy', 'clim_mean', 'clim_std']
    
    # Merge climatology
    result = result.merge(climatology, on=['Region', 'doy'], how='left')
    
    # Anomaly (absolute)
    result['rainfall_anomaly'] = result[rainfall_col] - result['clim_mean']
    
    # Standardized anomaly (z-score)
    result['rainfall_zscore'] = (
        result['rainfall_anomaly'] / result['clim_std'].replace(0, np.nan)
    ).fillna(0)
    
    # Percentile rank
    result['rainfall_percentile'] = (
        result.groupby(['Region', 'doy'])[rainfall_col]
        .transform(lambda x: x.rank(pct=True))
    )
    
    return result

rainfall_df = add_anomaly_features(rainfall_df)
print("Anomaly features added:")
print([c for c in rainfall_df.columns if 'anomaly' in c or 'zscore' in c or 'percentile' in c])

Anomaly features added:
['rainfall_anomaly', 'rainfall_zscore', 'rainfall_percentile']


## Region Encoding (for RL state)


In [12]:
# Create numeric region encoding
region_mapping = {r: i for i, r in enumerate(sorted(rainfall_df['Region'].unique()))}
rainfall_df['region_id'] = rainfall_df['Region'].map(region_mapping)

# One-hot encoding for regions
region_dummies = pd.get_dummies(rainfall_df['Region'], prefix='region')
rainfall_df = pd.concat([rainfall_df, region_dummies], axis=1)

print(f"Region mapping: {region_mapping}")

Region mapping: {'Region1': 0, 'Region2': 1, 'Region3': 2, 'Region4': 3, 'Region5': 4}


In [13]:
print(f"Total features: {len(rainfall_df.columns)}")
print(f"Total records: {len(rainfall_df):,}")
print(f"\nFeature categories:")

feature_categories = {
    'Original': ['Year', 'Month', 'Day', 'CHIRPS', 'TAMSAT', 'Mean', 'Region', 'Date'],
    'Rolling': [c for c in rainfall_df.columns if 'rainfall_sum' in c or 'rainfall_avg' in c or 'rainfall_max' in c or 'rainfall_std' in c or 'rainy_days_' in c],
    'Dry Spell': [c for c in rainfall_df.columns if 'dry' in c.lower() or 'since_rain' in c],
    'Season': ['onset_doy', 'cessation_doy', 'season_length', 'days_from_onset', 'days_to_cessation', 'in_season'],
    'Temporal': [c for c in rainfall_df.columns if 'doy' in c or 'month_' in c or 'dekad' in c or 'week' in c],
    'Lag': [c for c in rainfall_df.columns if 'lag' in c],
    'Cumulative': [c for c in rainfall_df.columns if 'cumulative' in c],
    'Anomaly': [c for c in rainfall_df.columns if 'anomaly' in c or 'zscore' in c or 'percentile' in c or 'clim' in c],
    'Region Encoding': [c for c in rainfall_df.columns if 'region' in c.lower()]
}

for cat, features in feature_categories.items():
    existing = [f for f in features if f in rainfall_df.columns]
    print(f"  {cat}: {len(existing)} features")

Total features: 60
Total records: 67,570

Feature categories:
  Original: 8 features
  Rolling: 15 features
  Dry Spell: 4 features
  Season: 6 features
  Temporal: 10 features
  Lag: 5 features
  Cumulative: 2 features
  Anomaly: 5 features
  Region Encoding: 7 features


In [14]:
# Missing value summary
print("Missing values per feature:")
missing = rainfall_df.isnull().sum()
missing_pct = (missing / len(rainfall_df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print(missing_df[missing_df['missing_count'] > 0].sort_values('missing_pct', ascending=False))

Missing values per feature:
                  missing_count  missing_pct
rainfall_lag_1y            1825         2.70
rainfall_lag_30d            150         0.22
rainfall_lag_14d             70         0.10
rainfall_lag_7d              35         0.05
rainfall_lag_1d               5         0.01


In [15]:
# Preview engineered features
feature_cols = ['Date', 'Region', 'Mean', 'rainfall_sum_7d', 'rainfall_avg_30d', 
                'dry_spell_length', 'days_since_rain_5mm', 'in_season', 
                'doy_sin', 'doy_cos', 'rainfall_anomaly', 'cumulative_season']
rainfall_df[feature_cols].head(15)

,Date,Region,Mean,rainfall_sum_7d,rainfall_avg_30d,dry_spell_length,days_since_rain_5mm,in_season,doy_sin,doy_cos,rainfall_anomaly,cumulative_season
0,1983-01-01,Region1,0.184418,0.184418,0.184418,1,1,0,0.017213,0.999852,-0.336457,0.0
1,1983-01-02,Region1,0.044112,0.228530,0.114265,2,2,0,0.034422,0.999407,-0.130212,0.0
2,1983-01-03,Region1,0.054328,0.282858,0.094286,3,3,0,0.051620,0.998667,-0.323442,0.0
3,1983-01-04,Region1,0.948458,1.231316,0.307829,4,4,0,0.068802,0.997630,0.154115,0.0
4,1983-01-05,Region1,0.842351,2.073667,0.414733,5,5,0,0.085965,0.996298,0.311521,0.0
5,1983-01-06,Region1,0.294867,2.368534,0.394756,6,6,0,0.103102,0.994671,-0.120042,0.0
6,1983-01-07,Region1,1.118112,3.486647,0.498092,0,7,0,0.120208,0.992749,0.695419,0.0
7,1983-01-08,Region1,0.120050,3.422279,0.450837,1,8,0,0.137279,0.990532,-0.160298,0.0
8,1983-01-09,Region1,0.278778,3.656945,0.431719,2,9,0,0.154309,0.988023,-0.164538,0.0
9,1983-01-10,Region1,0.165407,3.768023,0.405088,3,10,0,0.171293,0.985220,-0.197249,0.0


In [ ]:
RL_STATE_FEATURES = [
    # Temporal context
    'doy_sin', 'doy_cos', 'month_sin', 'month_cos',
    
    # Recent rainfall conditions
    'rainfall_sum_7d', 'rainfall_sum_14d', 'rainfall_sum_30d',
    'rainfall_avg_7d', 'rainfall_avg_30d',
    'rainy_days_7d', 'rainy_days_30d',
    
    # Drought risk indicators
    'dry_spell_length', 'max_dry_spell_30d', 'days_since_rain_5mm',
    
    # Season position
    'days_from_onset', 'days_to_cessation', 'in_season',
    
    # Cumulative progress
    'cumulative_season',
    
    # Context vs normal
    'rainfall_anomaly', 'rainfall_zscore',
    
    # Region (choose one encoding method)
    'region_id'  # or use one-hot: 'region_Region1', 'region_Region2', etc.
]

available_rl_features = [f for f in RL_STATE_FEATURES if f in rainfall_df.columns]
print(f"RL State Features Available: {len(available_rl_features)}/{len(RL_STATE_FEATURES)}")
print(available_rl_features)

RL State Features Available: 21/21
['doy_sin', 'doy_cos', 'month_sin', 'month_cos', 'rainfall_sum_7d', 'rainfall_sum_14d', 'rainfall_sum_30d', 'rainfall_avg_7d', 'rainfall_avg_30d', 'rainy_days_7d', 'rainy_days_30d', 'dry_spell_length', 'max_dry_spell_30d', 'days_since_rain_5mm', 'days_from_onset', 'days_to_cessation', 'in_season', 'cumulative_season', 'rainfall_anomaly', 'rainfall_zscore', 'region_id']


## Save Engineered Features


In [ ]:
output_path = DATA_PROCESSED / 'rainfall_features_full.csv'
rainfall_df.to_csv(output_path, index=False)
print(f"Saved full feature set: {output_path}")
print(f"  Shape: {rainfall_df.shape}")


rl_cols = ['Date', 'Year', 'Month', 'Day', 'Region', 'Mean'] + available_rl_features
rl_cols = list(dict.fromkeys(rl_cols))  # Remove duplicates, preserve order
rl_df = rainfall_df[rl_cols].copy()

rl_output_path = DATA_PROCESSED / 'rainfall_rl_features.csv'
rl_df.to_csv(rl_output_path, index=False)
print(f"\nSaved RL feature set: {rl_output_path}")
print(f"  Shape: {rl_df.shape}")

season_output_path = DATA_PROCESSED / 'season_dates.csv'
season_dates.to_csv(season_output_path, index=False)
print(f"\nSaved season dates: {season_output_path}")
print(f"  Shape: {season_dates.shape}")

Saved full feature set: ../data/processed/rainfall_features_full.csv
  Shape: (67570, 60)

Saved RL feature set: ../data/processed/rainfall_rl_features.csv
  Shape: (67570, 27)

Saved season dates: ../data/processed/season_dates.csv
  Shape: (185, 5)
